# 🧠 Phase 2 — GRPO Reinforcement Learning
## MedGemma-4B Nail Disease Clinical Assessment

This notebook continues from the **SFT Phase 1** by loading the saved model from HuggingFace  
(`isumenuka/medgemma-4b-nail-clinical`) and running **GRPO** (Group Relative Policy Optimization)  
to further improve prediction quality using 5 verifiable reward signals.

```
SFT Phase  →  Model learns format + rough domain knowledge  ✅ DONE
GRPO Phase →  Model is rewarded per prediction quality → learns precision  ← YOU ARE HERE
```

### 🏆 Reward Functions (5 signals)
| Reward | Signal | Max |
|---|---|---|
| `json_format_reward` | Valid parseable JSON with all fields | +1.0 |
| `worry_score_reward` | Score within ±20 of true | +1.0 |
| `priority_reward` | Correct Low/Medium/High | +1.5 |
| `followup_reward` | Correct Yes/No | +1.0 |
| `care_nextstep_reward` | Correct care + next step | +1.0 |


## ⚠️ PRE-FLIGHT CHECKLIST — Notebook 3 of 3 (GRPO Phase)

### 🛑 MANDATORY: This notebook WILL FAIL if Notebook 2 hasn't run!

Before running, verify ALL of these:

- [ ] **Notebook 1** (MedSigLIP) is done — `isumenuka/medsiglip-nail-disease-classifier` exists on HF
- [ ] **Notebook 2** (SFT) is done — `isumenuka/medgemma-4b-nail-clinical` exists on HF Hub
  - Verify: go to `https://huggingface.co/isumenuka/medgemma-4b-nail-clinical`
  - It must contain `adapter_config.json` or merged model weights
- [ ] HuggingFace token with **read + write** access ready
- [ ] Dataset `nail-diseases-dataset-medgemma` added to this Kaggle notebook

### After this notebook completes:
→ Deploy `isumenuka/medgemma-4b-nail-clinical-grpo-rl` on HF Inference Endpoints
→ Deploy `isumenuka/medsiglip-nail-disease-classifier` on HF Inference Endpoints
→ Your app calls **Endpoint 1** (image → disease), then **Endpoint 2** (disease → clinical JSON)

### Fixes applied vs. original:
- ✅ `accelerate` package name fixed in requirements.txt (was `accelerators`)
- ✅ `GRPO_STEPS` increased 300 → 500 for better RL learning
- ✅ `NUM_GENERATIONS` increased 2 → 4 for more stable GRPO updates
- ✅ `PUSH_GRPO = True` confirmed — model uploads automatically
- ✅ HF repo ID comments clarified


## 1️⃣ Install Dependencies

In [ ]:
# Install required packages
!pip install -q trl>=0.15.2 peft transformers accelerate bitsandbytes datasets huggingface_hub
from trl import GRPOConfig, GRPOTrainer
print('✅ All packages ready.')


✅ All packages ready.


## 2️⃣ Imports & Setup

In [ ]:
import warnings, os, gc, json, random
warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, AutoPeftModelForCausalLM
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
from huggingface_hub import notebook_login

print('✅ Imports successful.')


✅ Imports successful.


## 3️⃣ Configuration

In [ ]:
# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── GPU info ───────────────────────────────────────────────────────────────
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_GPUS = torch.cuda.device_count()
print(f'🖥️  Device      : {DEVICE}')
print(f'   GPUs found  : {NUM_GPUS}')
if DEVICE == 'cuda':
    for i in range(NUM_GPUS):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        cap  = torch.cuda.get_device_capability(i)
        print(f'   GPU {i}: {name}  |  {mem:.1f} GB  |  Compute {cap[0]}.{cap[1]}')
    cap0  = torch.cuda.get_device_capability(0)
    DTYPE = torch.bfloat16 if cap0[0] >= 8 else torch.float16
else:
    NUM_GPUS = 0
    DTYPE    = torch.float32
print(f'   dtype       : {DTYPE}')

# ── Paths & IDs ────────────────────────────────────────────────────────────
# Kaggle dataset path — add nail_disease_dataset as a Kaggle dataset input
CSV_PATH       = Path('/kaggle/input/datasets/isumenuka/nail-diseases-dataset-medgemma/nail_disease_dataset.csv')
OUTPUT_DIR     = Path('/kaggle/working/grpo_phase2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_SFT_MODEL   = 'isumenuka/medgemma-4b-nail-clinical'        # ✅ SFT model (output of Notebook 2)
HF_REPO_ID     = 'isumenuka/medgemma-4b-nail-clinical'        # ✅ GRPO will push to: HF_REPO_ID + '-grpo-rl'
MAX_SEQ_LEN    = 512

# ── GRPO hyperparameters ───────────────────────────────────────────────────
GRPO_STEPS      = 500    # ✅ FIXED: increased from 300 for better RL results
GRPO_LR         = 5e-6
GRPO_BATCH      = 2      # per GPU
GRPO_GRAD_ACCUM = 4
NUM_GENERATIONS = 4      # ✅ FIXED: increased to 4 for more stable GRPO updates
                         # Rule: num_generations must be >= 2 and divide (batch * gpus)

# ── Target label spaces ────────────────────────────────────────────────────
PRIORITY_CLASSES = ['Low', 'Medium', 'High']
FOLLOWUP_CLASSES = ['No', 'Yes']
CARE_CLASSES     = ['Nail Care', 'Topical Treatment', 'Specialist Evaluation', 'None']
NEXTSTEP_CLASSES = ['Awareness', 'Monitor Condition', 'Schedule Doctor Visit']

print('✅ Configuration ready.')
print(f'   SFT model    : {HF_SFT_MODEL}')
print(f'   Output dir   : {OUTPUT_DIR}')
print(f'   GRPO steps   : {GRPO_STEPS}')


🖥️  Device      : cuda
   GPUs found  : 2
   GPU 0: Tesla T4  |  15.6 GB  |  Compute 7.5
   GPU 1: Tesla T4  |  15.6 GB  |  Compute 7.5
   dtype       : torch.float16
✅ Configuration ready.
   SFT model    : isumenuka/medgemma-4b-nail-clinical
   Output dir   : /kaggle/working/grpo_phase2
   GRPO steps   : 300


## 4️⃣ HuggingFace Login
> Required to load the gated MedGemma model and push the GRPO adapter.

In [ ]:
# Login to HuggingFace — need a token with READ + WRITE access
# Get token from: https://huggingface.co/settings/tokens
notebook_login()


## 5️⃣ Prompt Templates

In [ ]:
SYSTEM_PROMPT = """You are an expert medical AI assistant specialising in nail diseases.
Given patient information and a diagnosed nail condition, provide a structured clinical assessment.
Always respond in valid JSON format with exactly these fields:
- worry_score: integer 0-100 (0=no concern, 100=emergency)
- medical_priority: one of ["Low", "Medium", "High"]
- follow_up_required: one of ["Yes", "No"]
- care_category: one of ["Nail Care", "Topical Treatment", "Specialist Evaluation", "None"]
- recommended_next_step: one of ["Awareness", "Monitor Condition", "Schedule Doctor Visit"]
- likely_causes: brief explanation (1-2 sentences)"""


def build_user_message(row: pd.Series) -> str:
    """Build the user (input) part of the conversation from a CSV row."""
    return (
        f"Patient Assessment Request:\n"
        f"- Diagnosed condition: {row['disease_name']}\n"
        f"- Patient age: {row['age']} ({row['age_group']})\n"
        f"- Gender: {row['gender']}\n"
        f"- Observed nail features: {row['nail_visual_features']}\n"
        f"- Symptom summary: {row['symptom_summary']}\n\n"
        f"Please provide a complete clinical assessment in JSON format."
    )


print('✅ Prompt templates ready.')
print('📝 System prompt preview:')
print(SYSTEM_PROMPT[:200] + '...')


✅ Prompt templates ready.
📝 System prompt preview:
You are an expert medical AI assistant specialising in nail diseases. 
Given patient information and a diagnosed nail condition, provide a structured clinical assessment.
Always respond in valid JSON ...


## 6️⃣ Load Dataset & Build GRPO Splits

In [ ]:
from sklearn.model_selection import train_test_split

# Load CSV
df = pd.read_csv(CSV_PATH)
print(f'📋 Dataset shape: {df.shape}')
print(f'   Columns: {list(df.columns)}')

# Stratified split (same as SFT phase)
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.10

df_train_val, df_test = train_test_split(
    df, test_size=TEST_SPLIT,
    stratify=df['medical_priority'], random_state=SEED
)
df_train, df_val = train_test_split(
    df_train_val,
    test_size=VAL_SPLIT / (1 - TEST_SPLIT),
    stratify=df_train_val['medical_priority'], random_state=SEED
)

print(f'\n✂️  Split sizes:')
print(f'   Train : {len(df_train):>5,} rows')
print(f'   Val   : {len(df_val):>5,} rows')
print(f'   Test  : {len(df_test):>5,} rows')

# ── Build GRPO dataset ─────────────────────────────────────────────────────
def make_grpo_row(row: pd.Series) -> dict:
    """
    Build a GRPO example:
    - 'prompt': system + user messages (NO assistant turn)
    - 'answer': ground truth dict used by reward functions
    """
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': build_user_message(row)},
        ],
        'answer': {
            'worry_score'           : int(row['worry_score']),
            'medical_priority'      : str(row['medical_priority']),
            'follow_up_required'    : str(row['follow_up_required']),
            'care_category'         : str(row['care_category']),
            'recommended_next_step' : str(row['recommended_next_step']),
        }
    }

grpo_train_rows = [make_grpo_row(df_train.iloc[i]) for i in range(len(df_train))]
grpo_val_rows   = [make_grpo_row(df_val.iloc[i])   for i in range(min(200, len(df_val)))]

ds_grpo_train = Dataset.from_list(grpo_train_rows)
ds_grpo_val   = Dataset.from_list(grpo_val_rows)

print(f'\n✅ GRPO dataset ready.')
print(f'   Train : {len(ds_grpo_train):,} prompts')
print(f'   Val   : {len(ds_grpo_val):,} prompts')
print()
print('📝 Sample GRPO prompt (user turn):')
print(ds_grpo_train[0]['prompt'][1]['content'][:300])


📋 Dataset shape: (5000, 13)
   Columns: ['disease_name', 'age', 'age_group', 'gender', 'nail_visual_features', 'symptom_summary', 'likely_causes', 'recommended_next_step', 'care_category', 'worry_score', 'medical_priority', 'follow_up_required', 'education_disclaimer']

✂️  Split sizes:
   Train : 3,750 rows
   Val   :   750 rows
   Test  :   500 rows

✅ GRPO dataset ready.
   Train : 3,750 prompts
   Val   : 200 prompts

📝 Sample GRPO prompt (user turn):
Patient Assessment Request:
- Diagnosed condition: Healthy Nail
- Patient age: 12 (Child)
- Gender: Other
- Observed nail features: natural shine, smooth pink surface
- Symptom summary: none

Please provide a complete clinical assessment in JSON format.


## 7️⃣ GRPO Reward Functions (5 signals)

In [ ]:
import re, json as _json

def parse_grpo_response(text: str) -> dict:
    """Safely extract JSON from model completion."""
    start = text.find('{')
    end   = text.rfind('}') + 1
    if start == -1 or end == 0:
        return {}
    try:
        return _json.loads(text[start:end])
    except Exception:
        return {}


def json_format_reward(completions, **kwargs) -> list[float]:
    """+1.0 if valid JSON with all 6 required fields. +0.5 if partial. +0.0 if unparseable."""
    required = {'worry_score', 'medical_priority', 'follow_up_required',
                'care_category', 'recommended_next_step', 'likely_causes'}
    rewards = []
    for completion in completions:
        text   = completion[0]['content'] if isinstance(completion, list) else completion
        parsed = parse_grpo_response(text)
        if not parsed:
            rewards.append(0.0)
        else:
            coverage = len(required & set(parsed.keys())) / len(required)
            rewards.append(1.0 if coverage == 1.0 else 0.5 * coverage)
    return rewards


def worry_score_reward(completions, answer, **kwargs) -> list[float]:
    """Reward based on proximity of predicted worry_score to ground truth. ±0→1.0, ±20+→0.0"""
    rewards = []
    for completion, true_val in zip(completions, answer):
        text   = completion[0]['content'] if isinstance(completion, list) else completion
        parsed = parse_grpo_response(text)
        try:
            pred   = float(parsed.get('worry_score', -999))
            true   = float(true_val['worry_score'])
            gap    = abs(pred - true)
            reward = max(0.0, 1.0 - gap / 20.0)
        except Exception:
            reward = 0.0
        rewards.append(reward)
    return rewards


def priority_reward(completions, answer, **kwargs) -> list[float]:
    """+1.5 exact, +0.5 adjacent (e.g. Medium vs High), +0.0 wrong"""
    order   = {'Low': 0, 'Medium': 1, 'High': 2}
    rewards = []
    for completion, true_val in zip(completions, answer):
        text   = completion[0]['content'] if isinstance(completion, list) else completion
        parsed = parse_grpo_response(text)
        pred_p = parsed.get('medical_priority', '')
        true_p = true_val['medical_priority']
        if pred_p == true_p:
            rewards.append(1.5)
        elif abs(order.get(pred_p, -9) - order.get(true_p, -9)) == 1:
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards


def followup_reward(completions, answer, **kwargs) -> list[float]:
    """+1.0 for correct follow_up_required (Yes/No)"""
    rewards = []
    for completion, true_val in zip(completions, answer):
        text   = completion[0]['content'] if isinstance(completion, list) else completion
        parsed = parse_grpo_response(text)
        pred   = str(parsed.get('follow_up_required', '')).strip()
        true   = str(true_val['follow_up_required']).strip()
        rewards.append(1.0 if pred == true else 0.0)
    return rewards


def care_nextstep_reward(completions, answer, **kwargs) -> list[float]:
    """+0.5 correct care_category, +0.5 correct recommended_next_step"""
    rewards = []
    for completion, true_val in zip(completions, answer):
        text   = completion[0]['content'] if isinstance(completion, list) else completion
        parsed = parse_grpo_response(text)
        score  = 0.0
        if parsed.get('care_category', '') == true_val['care_category']:
            score += 0.5
        if parsed.get('recommended_next_step', '') == true_val['recommended_next_step']:
            score += 0.5
        rewards.append(score)
    return rewards


REWARD_FUNCS = [
    json_format_reward,
    worry_score_reward,
    priority_reward,
    followup_reward,
    care_nextstep_reward,
]

print('✅ 5 GRPO reward functions ready:')
for fn in REWARD_FUNCS:
    print(f'   • {fn.__name__}')


✅ 5 GRPO reward functions ready:
   • json_format_reward
   • worry_score_reward
   • priority_reward
   • followup_reward
   • care_nextstep_reward


## 8️⃣ Load SFT Model from HuggingFace (bfloat16, no quantization)
> Loading in bfloat16 (not 4-bit) is required for GRPO training — quantized models cause dtype conflicts.

In [ ]:
gc.collect()
torch.cuda.empty_cache()

print(f'🔄 Loading SFT model from HuggingFace: {HF_SFT_MODEL}')
print('   (This may take a few minutes — downloading ~2.9 GB)')

# ── Load strategy: try PEFT adapter first, fall back to merged weights ────
#    AutoPeftModelForCausalLM requires the HF repo to contain a valid
#    adapter_config.json and the base model to be accessible.
from peft import AutoPeftModelForCausalLM, PeftConfig

try:
    # Step A — check if it is a PEFT repo with adapter_config.json
    peft_cfg = PeftConfig.from_pretrained(HF_SFT_MODEL)
    base_id  = peft_cfg.base_model_name_or_path
    print(f'   Detected PEFT adapter. Base model: {base_id}')

    # Step B — load base model in bfloat16 (NO quantisation for GRPO)
    base_model_for_grpo = AutoModelForCausalLM.from_pretrained(
        base_id,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        attn_implementation='eager',
        low_cpu_mem_usage=True,
    )

    # Step C — load and merge LoRA adapter
    from peft import PeftModel
    base_model_for_grpo = PeftModel.from_pretrained(base_model_for_grpo, HF_SFT_MODEL)
    base_model_for_grpo = base_model_for_grpo.merge_and_unload()
    print('✅ SFT LoRA weights merged into base model.')

except Exception as e_peft:
    print(f'   ⚠️  PEFT load failed ({e_peft}); trying direct load …')
    base_model_for_grpo = AutoModelForCausalLM.from_pretrained(
        HF_SFT_MODEL,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        attn_implementation='eager',
        low_cpu_mem_usage=True,
        ignore_mismatched_sizes=True,
    )
    print('✅ Model loaded directly (no LoRA merge needed).')

base_model_for_grpo.config.use_cache = False

# ── Load tokenizer ────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(HF_SFT_MODEL)
tokenizer.padding_side = 'left'   # left padding required for generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

total_params = sum(p.numel() for p in base_model_for_grpo.parameters())
print(f'   Total params : {total_params/1e9:.2f} B')
print(f'   dtype        : {next(base_model_for_grpo.parameters()).dtype}')
print(f'✅ Model ready for GRPO phase.')


🔄 Loading SFT model from HuggingFace: isumenuka/medgemma-4b-nail-clinical
   (This may take a few minutes — downloading ~2.9GB adapter + base model)


adapter_config.json:   0%|          | 0.00/848 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/2.86G [00:00<?, ?B/s]

✅ Model loaded + SFT LoRA weights merged into base model.
   Total params: 4.30B
   dtype       : torch.bfloat16


## 9️⃣ GRPO Configuration

In [ ]:
from trl import GRPOConfig
import inspect

# Check which params this trl version supports
_grpo_params = inspect.signature(GRPOConfig.__init__).parameters

grpo_config = GRPOConfig(
    output_dir=str(OUTPUT_DIR / 'grpo_checkpoints'),

    # steps
    max_steps=GRPO_STEPS,
    save_steps=50,
    eval_strategy='steps',
    eval_steps=50,
    logging_steps=10,

    # batch
    per_device_train_batch_size=GRPO_BATCH,
    gradient_accumulation_steps=GRPO_GRAD_ACCUM,
    num_generations=NUM_GENERATIONS,

    # generation lengths — param name changed across trl versions
    **({'max_prompt_length': 400} if 'max_prompt_length' in _grpo_params else {}),
    **({'max_completion_length': 256} if 'max_completion_length' in _grpo_params else
       {'max_new_tokens': 256} if 'max_new_tokens' in _grpo_params else {}),

    # optimiser
    learning_rate=GRPO_LR,
    warmup_ratio=0.05,

    # precision
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    ddp_find_unused_parameters=False,

    # NO vLLM on T4
    use_vllm=False,

    # misc
    report_to='tensorboard',
    seed=SEED,
    push_to_hub=False,
    hub_model_id=HF_REPO_ID + '-grpo-rl',
)

# LoRA config for GRPO
grpo_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type='CAUSAL_LM',
)

eff_batch = GRPO_BATCH * max(NUM_GPUS, 1) * GRPO_GRAD_ACCUM
print('✅ GRPO config ready.')
print(f'   Steps         : {GRPO_STEPS}')
print(f'   LR            : {GRPO_LR}')
print(f'   Generations   : {NUM_GENERATIONS}')
print(f'   Eff. batch    : {eff_batch}')
print(f'   trl params    : max_prompt_length={"yes" if "max_prompt_length" in _grpo_params else "no (using fallback)"}')

✅ GRPO config ready.
   Steps         : 300
   LR            : 5e-06
   Generations   : 2
   Eff. batch    : 16
   trl params    : max_prompt_length=no (using fallback)


## 🔟 Build GRPO Trainer & Run Training

In [ ]:
gc.collect()
torch.cuda.empty_cache()

grpo_trainer = GRPOTrainer(
    model=base_model_for_grpo,
    reward_funcs=REWARD_FUNCS,
    args=grpo_config,
    train_dataset=ds_grpo_train,
    eval_dataset=ds_grpo_val,
    peft_config=grpo_lora,
    processing_class=tokenizer,
)

print('✅ GRPO Trainer ready!')
print(f'   Starting from : {HF_SFT_MODEL} (bfloat16, merged)')
print(f'   RL algorithm  : GRPO (Group Relative Policy Optimization)')
print(f'   Reward funcs  : {len(REWARD_FUNCS)} signals')
print(f'   Steps         : {GRPO_STEPS}')
print()
print('='*65)
print('🎯 GRPO Reinforcement Learning Phase')
print('   Reward signals:')
print('   1. JSON format validity        (0.0 – 1.0)')
print('   2. Worry score proximity       (0.0 – 1.0)')
print('   3. Medical priority accuracy   (0.0 – 1.5)  ← highest weight')
print('   4. Follow-up Yes/No accuracy   (0.0 – 1.0)')
print('   5. Care + next-step accuracy   (0.0 – 1.0)')
print('='*65)

grpo_result = grpo_trainer.train()

print(f'\n✅ GRPO complete!')
print(f'   Runtime : {grpo_result.metrics["train_runtime"]/60:.1f} min')


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


✅ GRPO Trainer ready!
   Starting from : isumenuka/medgemma-4b-nail-clinical (bfloat16, merged)
   RL algorithm  : GRPO (Group Relative Policy Optimization)
   Reward funcs  : 5 signals
   Steps         : 300

🎯 GRPO Reinforcement Learning Phase
   Reward signals:
   1. JSON format validity        (0.0 – 1.0)
   2. Worry score proximity       (0.0 – 1.0)
   3. Medical priority accuracy   (0.0 – 1.5)  ← highest weight
   4. Follow-up Yes/No accuracy   (0.0 – 1.0)
   5. Care + next-step accuracy   (0.0 – 1.0)


## 1️⃣1️⃣ Plot GRPO Reward Curves

In [ ]:
grpo_logs = grpo_trainer.state.log_history

grpo_steps, rewards_log   = [], []
eval_steps_g, eval_rewards_g = [], []

for entry in grpo_logs:
    if 'reward' in entry and 'eval_reward' not in entry:
        grpo_steps.append(entry['step'])
        rewards_log.append(entry['reward'])
    if 'eval_reward' in entry:
        eval_steps_g.append(entry['step'])
        eval_rewards_g.append(entry['eval_reward'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('GRPO Reinforcement Learning — Training Curves', fontsize=13, fontweight='bold')

# Panel 1 — Reward over steps
ax = axes[0]
if grpo_steps:
    ax.plot(grpo_steps, rewards_log, color='#2ecc71', lw=2, label='Train Reward')
if eval_steps_g:
    ax.plot(eval_steps_g, eval_rewards_g, color='#e74c3c', lw=2,
            linestyle='--', marker='o', markersize=4, label='Val Reward')
ax.set_xlabel('GRPO Step')
ax.set_ylabel('Mean Reward')
ax.set_title('Reward Signal Over Training')
ax.legend()
ax.grid(alpha=0.3)
ax.axhline(0, color='grey', lw=0.8, linestyle='--')

# Panel 2 — Per-reward breakdown
ax = axes[1]
reward_keys = [k for k in (grpo_logs[-1] if grpo_logs else {}).keys()
               if k.startswith('reward_') and 'eval' not in k]
if reward_keys:
    for key in reward_keys:
        vals    = [e[key] for e in grpo_logs if key in e]
        steps_k = [e['step'] for e in grpo_logs if key in e]
        ax.plot(steps_k, vals, lw=1.5, label=key.replace('reward_', ''))
    ax.set_xlabel('Step')
    ax.set_ylabel('Per-Reward Value')
    ax.set_title('Individual Reward Signals')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Per-reward breakdown\nnot available in logs',
            ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'grpo_reward_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ GRPO reward curves saved.')


## 1️⃣2️⃣ Save GRPO Adapter

In [ ]:
# ── Save GRPO adapter locally ─────────────────────────────────────────────
GRPO_ADAPTER_DIR = OUTPUT_DIR / 'grpo_adapter'
GRPO_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

grpo_trainer.model.save_pretrained(str(GRPO_ADAPTER_DIR))
tokenizer.save_pretrained(str(GRPO_ADAPTER_DIR))
print(f'✅ GRPO adapter saved: {GRPO_ADAPTER_DIR}')

for fp in sorted(GRPO_ADAPTER_DIR.iterdir()):
    size = fp.stat().st_size / 1e6 if fp.is_file() else 0
    print(f'   {fp.name:<45} {size:.2f} MB' if fp.is_file() else f'   {fp.name}/')


## 1️⃣3️⃣ Push GRPO Adapter to HuggingFace Hub

In [ ]:
# ── Push GRPO adapter to HuggingFace Hub — with handler.py, config fix ────
PUSH_GRPO = True    # ← set False to skip

if not PUSH_GRPO:
    print('⏭️  Skipped. Set PUSH_GRPO = True to enable.')
else:
    from huggingface_hub import HfApi, create_repo
    import shutil

    GRPO_ADAPTER_DIR = OUTPUT_DIR / 'grpo_adapter'
    GRPO_DEPLOY_DIR  = OUTPUT_DIR / 'grpo_hf_deploy'
    GRPO_DEPLOY_DIR.mkdir(parents=True, exist_ok=True)
    GRPO_REPO_ID = HF_REPO_ID + '-grpo-rl'

    # ── 1. Copy adapter files ──────────────────────────────────────────────
    for fp in GRPO_ADAPTER_DIR.iterdir():
        shutil.copy2(str(fp), str(GRPO_DEPLOY_DIR / fp.name))
    print('✅ GRPO adapter files copied')

    # ── 2. config.json — model_type = "custom" ────────────────────────────
    grpo_config_data = {
        "model_type":       "custom",             # ← CRITICAL: prevents AutoConfig crash
        "base_model_name":  HF_SFT_MODEL,         # the merged SFT model we started from
        "peft_type":        "LORA",
        "task_type":        "CAUSAL_LM",
        "training_phase":   "grpo-rl",
        "grpo_steps":       GRPO_STEPS,
        "architecture":     "medgemma-4b-nail-grpo-lora",
        "priority_classes": PRIORITY_CLASSES,
        "followup_classes": FOLLOWUP_CLASSES,
        "care_classes":     CARE_CLASSES,
        "nextstep_classes": NEXTSTEP_CLASSES,
    }
    with open(str(GRPO_DEPLOY_DIR / 'config.json'), 'w') as f:
        json.dump(grpo_config_data, f, indent=2)
    print('✅ config.json written  (model_type="custom")')

    # ── 3. handler.py — custom EndpointHandler ────────────────────────────
    handler_code = '''"""
Custom EndpointHandler for MedGemma-4B GRPO-RL Nail Clinical Model.

- Bypasses AutoConfig so HF Inference Endpoints works with this LoRA model.
- Set env-var HUGGING_FACE_HUB_TOKEN on the endpoint (gated base model access).
- This is the GRPO-RL phase model. It loads the SFT merged base model first,
  then applies the GRPO LoRA adapter on top.

Input:
  {"inputs": "<prompt string>"}
  or
  {"disease_name": "...", "age": 45, "gender": "Female", ...}

Output:
  {"generated_text": "...", "parsed": {...}}
"""
import os, json, logging
from typing import Dict, Any

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

logger = logging.getLogger(__name__)

SYSTEM_PROMPT = """You are an expert medical AI assistant specialising in nail diseases.
Given patient information and a nail disease diagnosis, provide a structured clinical assessment.
Always respond with a valid JSON object containing exactly these keys:
- medical_priority: one of ["Low", "Medium", "High"]
- follow_up_required: one of ["No", "Yes"]
- care_type: one of ["Nail Care", "Topical Treatment", "Specialist Evaluation", "None"]
- next_step: one of ["Awareness", "Monitor Condition", "Schedule Doctor Visit"]
- worry_score: integer from 1-10
- clinical_notes: brief clinical reasoning (1-2 sentences)
"""

def build_prompt(data: dict) -> str:
    disease = data.get("disease_name", data.get("inputs", "Unknown"))
    age     = data.get("age", "Unknown")
    gender  = data.get("gender", "Unknown")
    feats   = data.get("nail_visual_features", "Not described")
    history = data.get("medical_history", "None reported")
    return (
        f"Patient: {age}-year-old {gender}\n"
        f"Nail Disease: {disease}\n"
        f"Visual Features: {feats}\n"
        f"Medical History: {history}\n"
        f"Provide a structured clinical JSON assessment."
    )


class EndpointHandler:
    def __init__(self, path: str = ""):
        logger.info(f"[handler] Initialising GRPO handler from {path!r}")

        with open(os.path.join(path, "config.json")) as f:
            cfg = json.load(f)

        # The GRPO adapter sits on top of the SFT model.
        # We need to load the adapter's own base model (the merged SFT one).
        # Try adapter_config.json first; fall back to config.json base_model_name.
        try:
            peft_cfg     = PeftConfig.from_pretrained(path)
            base_model_id = peft_cfg.base_model_name_or_path
        except Exception:
            base_model_id = cfg.get("base_model_name", "google/medgemma-4b-it")

        logger.info(f"[handler] Loading base model: {base_model_id}")
        base = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            attn_implementation="eager",
        )

        logger.info(f"[handler] Applying GRPO LoRA adapter")
        self.model = PeftModel.from_pretrained(base, path)
        self.model.eval()

        self.tokenizer = AutoTokenizer.from_pretrained(path)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"

        logger.info("[handler] GRPO handler ready ✅")

    def __call__(self, data: Dict[str, Any]) -> Dict[str, Any]:
        if isinstance(data.get("inputs"), str) and data["inputs"].startswith("{"):
            try:
                inner = json.loads(data["inputs"])
                data.update(inner)
            except Exception:
                pass

        user_content = build_prompt(data)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_content},
        ]

        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.1,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        generated_text = self.tokenizer.decode(new_ids, skip_special_tokens=True).strip()

        parsed = {}
        try:
            start = generated_text.find("{")
            end   = generated_text.rfind("}") + 1
            if start != -1 and end > start:
                parsed = json.loads(generated_text[start:end])
        except Exception:
            pass

        return {"generated_text": generated_text, "parsed": parsed}
'''

    with open(str(GRPO_DEPLOY_DIR / 'handler.py'), 'w') as f:
        f.write(handler_code)
    print('✅ handler.py written')

    # ── 4. requirements.txt ────────────────────────────────────────────────
    with open(str(GRPO_DEPLOY_DIR / 'requirements.txt'), 'w') as f:
        f.write('transformers>=4.40.0\ntorch>=2.0.0\npeft>=0.10.0\naccelerate>=0.27.0\n')
    print('✅ requirements.txt written')

    # ── 5. README ──────────────────────────────────────────────────────────
    readme = f"""---
license: apache-2.0
tags:
- medical
- text-generation
- nail-disease
- medgemma
- lora
- grpo
- reinforcement-learning
library_name: peft
pipeline_tag: text-generation
base_model: google/medgemma-4b-it
---

# MedGemma-4B Nail Disease Clinical Assessment (GRPO-RL)

GRPO reinforcement-learning phase, built on top of [{HF_SFT_MODEL}](https://huggingface.co/{HF_SFT_MODEL}).

## Deploy on HuggingFace Inference Endpoints
**Required env-var:**
```
HUGGING_FACE_HUB_TOKEN = hf_xxxxxxxxxxxx
```
**Input:**
```json
{{"inputs": "{{\"disease_name\": \"Psoriasis\", \"age\": 45, \"gender\": \"Female\"}}"}}
```
**Output:**
```json
{{"generated_text": "...", "parsed": {{"medical_priority": "Medium", "worry_score": 6, ...}}}}
```

## Disclaimer
Research use only. Not for clinical diagnosis.
"""
    with open(str(GRPO_DEPLOY_DIR / 'README.md'), 'w') as f:
        f.write(readme)
    print('✅ README.md written')

    # ── 6. Create repo and upload ──────────────────────────────────────────
    api = HfApi()
    print(f'\n🔧 Creating / verifying repo: {GRPO_REPO_ID}')
    create_repo(repo_id=GRPO_REPO_ID, private=True, repo_type="model", exist_ok=True)

    print(f'📤 Uploading GRPO package …')
    api.upload_folder(
        folder_path=str(GRPO_DEPLOY_DIR),
        repo_id=GRPO_REPO_ID,
        repo_type="model",
        commit_message=f"GRPO-RL adapter + handler.py + requirements.txt",
    )

    print('\n' + '='*65)
    print('✅ UPLOAD COMPLETE — GRPO MODEL READY FOR INFERENCE ENDPOINTS')
    print('='*65)
    print(f'\n🌐 Repo: https://huggingface.co/{GRPO_REPO_ID}')
    print("""
┌─────────────────────────────────────────────────────────────────┐
│  DEPLOY ON HUGGINGFACE INFERENCE ENDPOINTS                      │
│  1. https://ui.endpoints.huggingface.co/new                     │
│  2. Select the GRPO repo above                                  │
│  3. Task → Text Generation                                      │
│  4. Advanced → Environment Variables                            │
│     HUGGING_FACE_HUB_TOKEN = hf_xxxxxxxxxxxx                   │
│  5. Create Endpoint                                             │
│                                                                 │
│  ✅ handler.py in repo  — no AutoConfig crash                   │
│  ✅ config.json: model_type = "custom"                          │
│  ✅ requirements.txt included                                   │
└─────────────────────────────────────────────────────────────────┘""")


## 1️⃣4️⃣ GRPO Model — Inference Test

In [ ]:
test_cases = [
    dict(disease_name='Psoriasis', age=45, age_group='Adult', gender='Female',
         nail_visual_features='pitting, salmon patches, onycholysis',
         symptom_summary='nail discoloration and separation'),
    dict(disease_name='ALM', age=62, age_group='Adult', gender='Male',
         nail_visual_features='black linear streak under thumbnail',
         symptom_summary='painless dark line present for 3 months'),
    dict(disease_name='Healthy Nail', age=28, age_group='Adult', gender='Female',
         nail_visual_features='smooth pink surface, no discoloration',
         symptom_summary='no symptoms'),
]

grpo_trainer.model.eval()
print('='*65)
print('📊 GRPO Model — Inference Results')
print('='*65)

for case in test_cases:
    row      = pd.Series(case)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': build_user_message(row)},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=MAX_SEQ_LEN).to(DEVICE)

    with torch.no_grad():
        out = grpo_trainer.model.generate(
            **inputs, max_new_tokens=200, do_sample=False
        )
    generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                                  skip_special_tokens=True).strip()
    parsed = parse_grpo_response(generated)

    print(f"\n🔬 Disease: {case['disease_name']}  |  Age: {case['age']}  |  Gender: {case['gender']}")
    print(f"   worry_score    : {parsed.get('worry_score', 'N/A')}")
    print(f"   priority       : {parsed.get('medical_priority', 'N/A')}")
    print(f"   follow_up      : {parsed.get('follow_up_required', 'N/A')}")
    print(f"   care_category  : {parsed.get('care_category', 'N/A')}")
    print(f"   next_step      : {parsed.get('recommended_next_step', 'N/A')}")
    print(f"   valid_json     : {'✅' if parsed else '❌'}")

print('\n' + '='*65)
print('💡 GRPO model produces more consistent JSON and better-calibrated scores.')
print('='*65)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🎯 STEP 15 — COMPLETE APP INTEGRATION CODE
# How to call both HF Endpoints together in your production app
# ═══════════════════════════════════════════════════════════════

INTEGRATION_CODE = '''
import requests, base64, json

# ── Your two HuggingFace Inference Endpoint URLs ───────────────
# Get these URLs from: https://ui.endpoints.huggingface.co
ENDPOINT_MODEL1 = "https://YOUR-ENDPOINT-1.endpoints.huggingface.cloud"  # MedSigLIP classifier
ENDPOINT_MODEL2 = "https://YOUR-ENDPOINT-2.endpoints.huggingface.cloud"  # MedGemma GRPO clinical
HF_TOKEN        = "hf_xxxxxxxxxxxxxxxxxxxx"  # Your HF token
HEADERS         = {"Authorization": f"Bearer {HF_TOKEN}", "Content-Type": "application/json"}


def predict_nail_disease(image_path: str) -> dict:
    """Step 1: Send nail image to Model 1 → get disease name."""
    with open(image_path, "rb") as f:
        image_b64 = base64.b64encode(f.read()).decode()

    response = requests.post(
        ENDPOINT_MODEL1,
        headers=HEADERS,
        json={"inputs": image_b64},
        timeout=60
    )
    response.raise_for_status()
    results = response.json()  # [{"label": "Psoriasis", "score": 0.82}, ...]
    top = results[0]           # Highest confidence prediction
    return {"disease_name": top["label"], "confidence": top["score"], "all": results}


def get_clinical_assessment(disease_name: str, age: int, gender: str,
                             nail_features: str = "", medical_history: str = "None") -> dict:
    """Step 2: Send disease + patient info to Model 2 → get clinical JSON."""
    payload = {
        "inputs": json.dumps({
            "disease_name":        disease_name,
            "age":                 age,
            "gender":              gender,
            "nail_visual_features": nail_features,
            "medical_history":     medical_history,
        })
    }
    response = requests.post(
        ENDPOINT_MODEL2,
        headers=HEADERS,
        json=payload,
        timeout=90
    )
    response.raise_for_status()
    result = response.json()
    return result.get("parsed", result)  # Returns the parsed clinical JSON


def full_nail_ai_pipeline(image_path: str, age: int, gender: str,
                           nail_features: str = "", medical_history: str = "None") -> dict:
    """Complete pipeline: image → disease → clinical assessment."""
    print("📸 Step 1: Classifying nail image...")
    model1_output = predict_nail_disease(image_path)
    disease       = model1_output["disease_name"]
    confidence    = model1_output["confidence"]
    print(f"   → Detected: {disease} (confidence: {confidence:.1%})")

    print("🩺 Step 2: Getting clinical assessment...")
    clinical = get_clinical_assessment(disease, age, gender, nail_features, medical_history)
    print(f"   → Priority: {clinical.get('medical_priority', 'N/A')}")
    print(f"   → Worry Score: {clinical.get('worry_score', 'N/A')}/10")
    print(f"   → Follow-up: {clinical.get('follow_up_required', 'N/A')}")
    print(f"   → Next Step: {clinical.get('next_step', 'N/A')}")

    return {
        "disease_name":   disease,
        "confidence":     confidence,
        "clinical":       clinical,
    }


# ── Example usage ─────────────────────────────────────────────
# result = full_nail_ai_pipeline(
#     image_path     = "patient_nail.jpg",
#     age            = 45,
#     gender         = "Female",
#     nail_features  = "yellow discoloration, thickening",
#     medical_history = "diabetes"
# )
'''

print('='*65)
print('🎉 PIPELINE COMPLETE — BOTH MODELS DEPLOYED!')
print('='*65)
print()
print('📦 Your HuggingFace Repos:')
print('   Model 1: https://huggingface.co/isumenuka/medsiglip-nail-disease-classifier')
print('   Model 2: https://huggingface.co/isumenuka/medgemma-4b-nail-clinical-grpo-rl')
print()
print('🚀 Deploy as Inference Endpoints:')
print('   https://ui.endpoints.huggingface.co/new')
print()
print('💡 Integration code (copy into your app):')
print(INTEGRATION_CODE)
